In [28]:
import os
import re
import pandas as pd
import numpy as np
import math
from scipy.optimize import minimize

In [2]:
input_file='data.MOF_5_opt_2x2x2'

In [3]:
with open(input_file, 'r') as file:
    lines = file.readlines()

# Functions

In [4]:
def get_unique_folder_name(input_file):
    base_name = 'output_' + input_file.replace('data','').replace('.','')
    folder_name = base_name
    counter = 1
    
    while os.path.exists(folder_name):
        folder_name = f"output_{counter}_{input_file.replace('data','').replace('.','')}"
        counter += 1
    
    os.makedirs(folder_name)
    return folder_name

folder_name=get_unique_folder_name(input_file)

In [5]:
def extract_numbers(text):
    # Extract all numbers including decimals and negatives
    numbers = re.findall(r'-?\d+\.?\d*', text)
    return [float(num) if '.' in num else int(num) for num in numbers]

In [6]:
def get_params_count():
    with open(input_file, 'r') as file:
        lines = file.readlines()
    
    comment=lines[0]
    
    atom_count=extract_numbers(lines[2])[0]
    bond_count=extract_numbers(lines[3])[0]
    angle_count=extract_numbers(lines[4])[0]
    dihedral_count=extract_numbers(lines[5])[0]
    improper_count=extract_numbers(lines[6])[0]

    return atom_count, bond_count, angle_count, dihedral_count, improper_count

atom_count, bond_count, angle_count, dihedral_count, improper_count = get_params_count()

In [7]:
def get_types_number():
    with open(input_file, 'r') as file:
            lines = file.readlines()
    
    for each in lines: #this returns the number of atom in atom_types, bonds in bond_types... angle_types, dihedral_types and improper_types
        if 'types' in each:
            line=each.split()
            variable=line[1]+'_'+line[2]
            value=line[0]
            globals()[variable]=int(value)


get_types_number()

In [8]:
headers=['Masses','Pair Coeffs','Bond Coeffs','Angle Coeffs','Dihedral Coeffs','Improper Coeffs','Atoms','Bonds','Angles','Dihedrals','Impropers']

def identify_headers_dict(lines, headers, default=-1):
    headers_dict = {}

    for i, line in enumerate(lines):
        for header in headers:
            if header not in headers_dict and header in line:
                headers_dict[header] = i
                if len(headers_dict) == len(headers):
                    return headers_dict

    # Add default values for missing headers
    for header in headers:
        if header not in headers_dict:
            headers_dict[header] = default

    return headers_dict

headers_dict = identify_headers_dict(lines, headers)



In [9]:
element_data=pd.read_csv( 'element_data.csv')

element_masses = element_data['A'].astype(float).values
element_names = element_data['element'].values

element_map = dict(zip(element_data['element'], element_data['Z']))

def find_closest_element(mass):
    idx = np.argmin(np.abs(element_masses - mass))
    return element_names[idx]



In [10]:
ff_types = pd.DataFrame({
    'type_number': [0] * atom_types,
    'masses': [None] * atom_types,
    'element': [None] * atom_types,
    'type': [None] * atom_types,
    'gro_sigma': [0.0] * atom_types,
    'gro_epsilon': [0.0] * atom_types
})

start_line = headers_dict['Masses'] + 2  # +1 for header, +1 for empty line

for i in range(atom_types):
    atom_data = lines[start_line + i].split()
    ff_types.loc[i, 'type_number'] = int(atom_data[0])
    ff_types.loc[i, 'masses'] = float(atom_data[1])
    ff_types.loc[i, 'element'] = find_closest_element(float(atom_data[1]))

    try:
        ff_types.loc[i, 'type'] = atom_data[3]
    except:
        ff_types.loc[i, 'type'] = ff_types.loc[i, 'element'].replace(' ','')+'_'+str(ff_types.loc[i, 'type_number'])

ff_types['atomic_number'] = ff_types['element'].map(element_map)
#========= STARTING PAIR COEFFS =================================
#gromacs uses kj/mol and nm while lammps uses kcal/mol and angstroms
#order on lammps coeff1= epsilon, coeff2=sigma

start_line = headers_dict['Pair Coeffs'] + 2  # +1 for header, +1 for empty line

for i in range(atom_types):
    atom_data = lines[start_line + i].split()


    ff_types.loc[i, 'gro_sigma'] = float(atom_data[2])/10           #A to nm
    ff_types.loc[i, 'gro_epsilon'] = float(atom_data[1])*4.184   #kcal/mol to kj/mol


In [11]:
def get_atom_data():
    lammps_data = np.loadtxt(input_file,
                         skiprows=headers_dict['Atoms'] + 2,
                         max_rows=atom_count)

    column_names = ['atom_id', 'mol_id', 'atom_type', 'charge', 'x', 'y', 'z','vel_x','vel_y','vel_z']

    atom_data = pd.DataFrame(lammps_data, columns=column_names[:lammps_data.shape[1]])
    element_mapping = dict(zip(ff_types['type_number'], ff_types['element']))
    type_mapping = dict(zip(ff_types['type_number'], ff_types['type']))

    atom_data['element'] = atom_data['atom_type'].map(element_mapping)
    atom_data['gro_type'] = atom_data['atom_type'].map(type_mapping)
    return atom_data

atom_data=get_atom_data()

In [12]:
def extract_box_params():
    xy, xz, yz = 0,0,0
    for line in lines:
        if "xlo xhi" in line:
            xlo, xhi = extract_numbers(line)
            lx = xhi - xlo
        if "ylo yhi" in line:
            ylo, yhi = extract_numbers(line)
            ly = yhi - ylo
        if "zlo zhi" in line:
            zlo, zhi = extract_numbers(line)
            lz = zhi - zlo
        if "xy xz yz" in line:
            xy, xz, yz = extract_numbers(line)
            break

    a=lx
    b=(ly**2+xy**2)**(1/2)
    c=(lz**2+xz**2+yz**2)**(1/2)

    alpha = (math.acos(((xy*xz)+(ly*yz))/(b*c)))*57.2958
    beta = (math.acos(xz/c))*57.2958
    gamma = (math.acos(xy/b))*57.2958

    gromacs_last_line=f'    {lx/10:.5f} {ly/10:.5f} {lz/10:.5f} {0:.5f} {0:.5f} {xy/10:.5f} {0/10:.5f} {xz/10:.5f} {yz/10:.5f}'

    return gromacs_last_line



In [ ]:
def get_bond_data():
    lammps_data = np.loadtxt(input_file,
                         skiprows=headers_dict['Bonds'] + 2,
                         max_rows=bond_count)

    column_names = ['bond_id', 'bond_type', 'ai', 'aj']

    bond_data = pd.DataFrame(lammps_data, columns=column_names[:lammps_data.shape[1]])

    atom_nr_2_type = dict(zip(atom_data['atom_id'], atom_data['gro_type'])) # TODO criar helper function pra isso

    bond_data['element_i'] = bond_data['ai'].map(atom_nr_2_type)
    bond_data['element_j'] = bond_data['aj'].map(atom_nr_2_type)

    bond_types_coeffs = bond_data[['bond_type', 'element_i', 'element_j']].drop_duplicates() #  TODO passar pro bond coeffs dps
    bond_types_coeffs = bond_types_coeffs.sort_values(by='bond_type').reset_index(drop=True)

    return bond_data,bond_types_coeffs

bond_data,bond_types_coeffs=get_bond_data()

In [ ]:
def get_bond_coeffs(headers_dict, bond_types):
    lammps_data = np.loadtxt(input_file,
                             skiprows=headers_dict['Bond Coeffs'] + 2,
                             max_rows=bond_types)

    column_names=['bond_type','kb_lammps','dist_lammps']

    bond_coeffs=pd.DataFrame(lammps_data, columns=column_names[:lammps_data.shape[1]])

    bond_types_coeffs['kb_gro']=bond_coeffs['kb_lammps']*4.184*100    #*2 #gromacs uses k/2 on its equation
    bond_types_coeffs['dist_gro']=bond_coeffs['dist_lammps']/10

get_bond_coeffs()

In [15]:
def get_angle_data():
    lammps_data = np.loadtxt(input_file,
                         skiprows=headers_dict['Angles'] + 2,
                         max_rows=angle_count)

    column_names = ['angle_id', 'angle_type', 'ai', 'aj', 'ak']

    angle_data = pd.DataFrame(lammps_data, columns=column_names[:lammps_data.shape[1]])

    atom_nr_2_type = dict(zip(atom_data['atom_id'], atom_data['gro_type']))

    angle_data['element_i'] = angle_data['ai'].map(atom_nr_2_type)
    angle_data['element_j'] = angle_data['aj'].map(atom_nr_2_type)
    angle_data['element_k'] = angle_data['ak'].map(atom_nr_2_type)

    angle_types_coeffs = angle_data[['angle_type', 'element_i', 'element_j','element_k']].drop_duplicates()
    angle_types_coeffs = angle_types_coeffs.sort_values(by='angle_type').reset_index(drop=True)

    return angle_data,angle_types_coeffs

angle_data,angle_types_coeffs=get_angle_data()

In [ ]:


def get_lammps_cos_per_params(C, B, n):                    # ========for cossine periodic angles===========
    theta = np.linspace(0, 2 * np.pi, 1000)
    theta_angle = theta * 180 / np.pi

    if B == 1:
        periodic = 3   
    else:
        periodic = 2

    theta0 = 180 / n * (periodic)

    E_list = []
    for i in range(len(theta)):
        E = 2 / (n**2) * C * (1 - (B * (-1**n) * (np.cos(n * theta[i]))))
        E_list.append(E)

    # Define the range for fitting
    fit_range_start = theta0 - 100/n
    fit_range_end = theta0 + 100/n

    # Filter data within the fitting range
    theta_fit = theta_angle[(theta_angle >= fit_range_start) & (theta_angle <= fit_range_end)]
    E_fit = np.array(E_list)[(theta_angle >= fit_range_start) & (theta_angle <= fit_range_end)]

    # Define the objective function to minimize (sum of squared differences)
    def objective_function(keff):
        E_harm_fit = keff * (theta_fit - theta0)**2
        return np.sum((E_fit - E_harm_fit)**2)

    # Find the optimal keff using optimization
    initial_keff = C / 4000  # Starting point for optimization
    result = minimize(objective_function, initial_keff)
    optimal_keff = result.x[0]

    return optimal_keff, theta0

def get_fourier_gromos_params(K, C0, C1, C2):
    theta = np.linspace(0, 2 * np.pi, 1000)
    theta_angle = theta * 180 / np.pi

    E_list = []
    for i in range(len(theta)):
        E = K * (C0 + C1 * (np.cos(theta[i])) + C2 * (np.cos(2 * theta[i])))
        E_list.append(E)

    # For the Fourier potential, the equilibrium angle is where the derivative is zero.
    # Finding the exact equilibrium angle can be complex, so we'll use the minimum of the calculated energies as a proxy for theta0.
    min_energy_index = np.argmin(E_list)
    theta0_approx = theta_angle[min_energy_index]

    # Define a fitting range around the approximate equilibrium angle
    # Adjust the range based on the expected width of the potential well
    # For simplicity here, we'll assume fitting over the whole range for now, similar to the previous function.
    theta_fit = theta_angle
    E_fit = np.array(E_list)


    # Convert theta_fit back to radians for the new equation
    theta_fit_rad = theta_fit * np.pi / 180

    # Define the objective function to minimize (sum of squared differences)
    
    def objective_function(params):
        k, theta_eq_rad = params
        # Ensure theta_eq_rad is within a valid range for np.cos
        theta_eq_rad = np.arctan2(np.sin(theta_eq_rad), np.cos(theta_eq_rad))

        E_new_eq_fit = k/2 * (np.cos(theta_fit_rad) - np.cos(theta_eq_rad))**2
        return np.sum((E_fit - E_new_eq_fit)**2)

    # Find the optimal k and theta_eq using optimization
    # Initial guess for k and theta_eq (in radians)
    initial_k = K / 1000  # Starting point for optimization, adjust as needed
    initial_theta_eq_rad = theta0_approx * np.pi / 180 # Starting point for optimization

    initial_params = [initial_k, initial_theta_eq_rad]

    # Use bounds to keep theta_eq_rad within a reasonable range (e.g., -2*pi to 2*pi) to help the optimizer
    bounds = [(None, None), (-2 * np.pi, 2 * np.pi)]


    result = minimize(objective_function, initial_params, bounds=bounds)
    optimal_k, optimal_theta_eq_rad = result.x
    optimal_theta_eq_angle = (optimal_theta_eq_rad * 180 / np.pi) % 360 # Ensure angle is within 0-360

    

    return optimal_k, optimal_theta_eq_angle

In [17]:
def get_angle_coeffs():
    lammps_data = np.loadtxt(input_file,
                             skiprows=headers_dict['Angle Coeffs'] + 2,
                             max_rows=angle_types)

    if len(lammps_data.T) == 4:
        angle_style='cossine-periodic'
        column_names=['angle_type','C_lammps','B_lammps','n_lammps']
        angle_coeffs=pd.DataFrame(lammps_data, columns=column_names[:lammps_data.shape[1]])

        angle_coeffs[['k_eff_lammps', 'theta0_deg']] = angle_coeffs.apply(
            lambda row: get_lammps_cos_per_params(C=row['C_lammps'],
                                          B=row['B_lammps'],
                                          n=row['n_lammps']), axis=1, result_type="expand")
        

    elif len(lammps_data.T) == 5:
        angle_style='fourier'
        column_names=['angle_type','k_lammps','C0_lammps','C1_lammps','C2_lammps']
        angle_coeffs=pd.DataFrame(lammps_data, columns=column_names[:lammps_data.shape[1]])

        angle_coeffs[['k_eff_lammps', 'theta0_deg']] = angle_coeffs.apply(
            lambda row: get_fourier_gromos_params(K=row['k_lammps'],
                                          C0=row['C0_lammps'],
                                          C1=row['C1_lammps'],
                                          C2=row['C2_lammps']), axis=1, result_type="expand")

    elif len(lammps_data.T) == 3:
        if lammps_data.ndim == 1:        # single row (1D)
            lammps_data = [lammps_data.tolist()]
        else:                            # already 2D
            lammps_data = lammps_data.tolist()
        angle_style = 'harmonic'
        column_names = ['angle_type', 'k_eff_lammps', 'theta0_deg']
        angle_coeffs=pd.DataFrame(lammps_data, columns=column_names)
        

    
    angle_coeffs['k_eff_gromacs']=angle_coeffs['k_eff_lammps']*4.184

    return angle_coeffs,angle_style

angle_coeffs,angle_style=get_angle_coeffs()

In [18]:
def get_dihedral_data():
    lammps_data = np.loadtxt(input_file,
                         skiprows=headers_dict['Dihedrals'] + 2,
                         max_rows= dihedral_count)

    column_names = ['dihedral_id', 'dihedral_type', 'ai', 'aj', 'ak','al']

    dihedral_data = pd.DataFrame(lammps_data, columns=column_names[:lammps_data.shape[1]])

    atom_nr_2_type = dict(zip(atom_data['atom_id'], atom_data['gro_type']))

    dihedral_data['element_i'] = dihedral_data['ai'].map(atom_nr_2_type)
    dihedral_data['element_j'] = dihedral_data['aj'].map(atom_nr_2_type)
    dihedral_data['element_k'] = dihedral_data['ak'].map(atom_nr_2_type)
    dihedral_data['element_l'] = dihedral_data['al'].map(atom_nr_2_type)

    dihedral_types_coeffs =  dihedral_data[['dihedral_type', 'element_i', 'element_j','element_k','element_l']].drop_duplicates()
    dihedral_types_coeffs =  dihedral_types_coeffs.sort_values(by='dihedral_type').reset_index(drop=True)

    return  dihedral_data, dihedral_types_coeffs

if dihedral_count != 0:
    dihedral_data, dihedral_types_coeffs=get_dihedral_data()

In [19]:
def get_improper_data():
    lammps_data = np.loadtxt(input_file,
                         skiprows=headers_dict['Impropers'] + 2,
                         max_rows= improper_count)

    column_names = ['improper_id', 'improper_type', 'ai', 'aj', 'ak','al']

    improper_data = pd.DataFrame(lammps_data, columns=column_names[:lammps_data.shape[1]])

    atom_nr_2_type = dict(zip(atom_data['atom_id'], atom_data['gro_type']))

    improper_data['element_i'] = improper_data['ai'].map(atom_nr_2_type)
    improper_data['element_j'] = improper_data['aj'].map(atom_nr_2_type)
    improper_data['element_k'] = improper_data['ak'].map(atom_nr_2_type)
    improper_data['element_l'] = improper_data['al'].map(atom_nr_2_type)

    improper_types_coeffs =  improper_data[['improper_type', 'element_i', 'element_j','element_k','element_l']].drop_duplicates()
    improper_types_coeffs =  improper_types_coeffs.sort_values(by='improper_type').reset_index(drop=True)

    return  improper_data, improper_types_coeffs

if improper_count !=0:
    improper_data, improper_types_coeffs=get_improper_data()

In [20]:
def get_dihedral_coeffs(dihedral_types_coeffs):
    lammps_data = np.loadtxt(input_file,
                             skiprows=headers_dict['Dihedral Coeffs'] + 2,
                             max_rows=dihedral_types)

    column_names=['dihedral_type','K_lammps','d_lammps','n_lammps']
    dihedral_coeffs=pd.DataFrame(lammps_data, columns=column_names[:lammps_data.shape[1]])

    dihedral_coeffs['K_gromacs']=dihedral_coeffs['K_lammps']*4.184
    dihedral_coeffs['phi_s_gromacs'] = np.where(dihedral_coeffs['d_lammps'] == 1, 0, 180)
    dihedral_coeffs['n_gromacs'] = dihedral_coeffs['n_lammps']

    # Merge relevant columns from dihedral_coeffs into dihedral_types_coeffs
    dihedral_types_coeffs = dihedral_types_coeffs.merge(
    dihedral_coeffs[['dihedral_type', 'K_gromacs', 'phi_s_gromacs', 'n_gromacs']],
    on='dihedral_type',
    how='left'  # Use 'left' to preserve existing rows in dihedral_types_coeffs
    )

    return dihedral_types_coeffs
    
if dihedral_count != 0:
    dihedral_types_coeffs=get_dihedral_coeffs(dihedral_types_coeffs)

In [ ]:
def get_fourier_harmonic_params(K_fourier, C0, C1, C2):
    """
    Calculates optimal parameters (k and theta0) for a harmonic potential
    by fitting it to a LAMMPS Fourier potential within the 0 to 180 degree interval.

    Args:
        K_fourier (float): LAMMPS Fourier K parameter.
        C0 (float): LAMMPS Fourier C0 parameter.
        C1 (float): LAMMPS Fourier C1 parameter.
        C2 (float): LAMMPS Fourier C2 parameter.

    Returns:
        tuple: Optimal k and theta0 for the harmonic potential.
    """
    theta = np.linspace(0, 2 * np.pi, 1000)
    theta_angle = theta * 180 / np.pi

    # Calculate LAMMPS Fourier potential
    E_fourier_list = []
    for i in range(len(theta)):
        E = K_fourier * (C0 + C1 * (np.cos(theta[i])) + C2 * (np.cos(2 * theta[i])))
        E_fourier_list.append(E)

    # Filter data to the 0 to 180 degree interval
    fit_range_start = 0
    fit_range_end = 90

    theta_fit = theta_angle[(theta_angle >= fit_range_start) & (theta_angle <= fit_range_end)]
    E_fit = np.array(E_fourier_list)[(theta_angle >= fit_range_start) & (theta_angle <= fit_range_end)]

    # Fix theta0 to 0
    theta0_fixed = 0

    # Define the objective function to minimize (sum of squared differences)
    # This function now only takes k as a parameter
    def objective_function(k):
        E_harmonic_fit = k/2 * (theta_fit - theta0_fixed)**2
        return np.sum((E_fit - E_harmonic_fit)**2)

    # Find the optimal k using optimization
    initial_k = K_fourier / 1000  # Example initial guess

    # Use bounds for k to be non-negative
    bounds = [(0, None)]

    result = minimize(objective_function, initial_k, bounds=bounds)
    optimal_k = result.x[0]*4.184  # Convert to GROMACS units

    return optimal_k, theta0_fixed

In [22]:
def get_improper_coeffs(improper_types_coeffs):
   
    lammps_data = np.loadtxt(input_file,
                             skiprows=headers_dict['Improper Coeffs'] + 2,
                             max_rows=improper_types)

    if lammps_data.ndim == 1:        # single row (1D)
        lammps_data = [lammps_data.tolist()]

    column_names=['improper_type','k_lammps','C0_lammps','C1_lammps','C2_lammps','all']
    try:
        improper_coeffs=pd.DataFrame(lammps_data, columns=column_names[:lammps_data.shape[1]])
    except:
        improper_coeffs=pd.DataFrame(lammps_data, columns=column_names)
        print(improper_coeffs)

    improper_coeffs[['k_eff_lammps', 'theta0_deg']] =improper_coeffs.apply(
        lambda row: get_fourier_harmonic_params(K_fourier=row['k_lammps'],
                                      C0=row['C0_lammps'],
                                      C1=row['C1_lammps'],
                                      C2=row['C2_lammps']), axis=1, result_type="expand")
    
    improper_coeffs['k_eff_gromacs']=improper_coeffs['k_eff_lammps']*4.184

    improper_types_coeffs = improper_types_coeffs.merge(
    improper_coeffs[['improper_type', 'k_eff_gromacs', 'theta0_deg']],
    on='improper_type',
    how='left'  # Use 'left' to preserve existing rows in dihedral_types_coeffs
    )
    
    return improper_types_coeffs
    
if improper_count !=0:
    improper_types_coeffs=get_improper_coeffs(improper_types_coeffs)

In [23]:
def write_bonded_info(ffbonded_name=f'{folder_name}/ffbonded.itp',angle_style=angle_style):
    if angle_style=='fourier':
        ang_num=2
    else:
        ang_num=1
    
    with open(ffbonded_name, 'w') as file:
        # Write header
        file.write(f''';Created using lmp2gro

[ bondtypes ]
;     i       j     func           b0                 kb
''')

        # Write atom types data
        for i in range(bond_types):
            line = (f"{bond_types_coeffs['element_i'][i]:>8}"
                    f"{bond_types_coeffs['element_j'][i]:>8}"
                    f"     1 "
                    f"{bond_types_coeffs['dist_gro'][i]:>20.6f}"
                    f"{bond_types_coeffs['kb_gro'][i]:>20.6f}\n")
            file.write(line)

        file.write(f'''
[ angletypes ]
;      i       j       k  func                  th0                cth
''')
        for i in range(angle_types):
            line = (f"{angle_types_coeffs['element_i'][i]:>8}"
                    f"{angle_types_coeffs['element_j'][i]:>8}"
                    f"{angle_types_coeffs['element_k'][i]:>8}"
                    f"     {ang_num} "
                    f"{angle_coeffs['theta0_deg'][i]:>20.6f}"
                    f"{angle_coeffs['k_eff_gromacs'][i]:>20.6f}\n")
            file.write(line)

        if dihedral_count !=0:
            file.write(f'''
[ dihedraltypes ]
;      i       j       k       l  func    phi_s               K        n
''')
            for i in range(len(dihedral_types_coeffs)):
                line = (f"{dihedral_types_coeffs['element_i'][i]:>8}"
                        f"{dihedral_types_coeffs['element_j'][i]:>8}"
                        f"{dihedral_types_coeffs['element_k'][i]:>8}"
                        f"{dihedral_types_coeffs['element_l'][i]:>8}"
                        f"     9 "
                        f"{dihedral_types_coeffs['phi_s_gromacs'][i]:>8.1f}"
                        f"{dihedral_types_coeffs['K_gromacs'][i]:>20.6f}"
                        f"{dihedral_types_coeffs['n_gromacs'][i]:>5.0f}\n")
                file.write(line)
            
        if improper_count !=0:
            file.write(f'''
[ dihedraltypes ]
;      i       j       k       l  func    thetha              K    
''')
            for i in range(len(improper_types_coeffs)):
                line = (f"{improper_types_coeffs['element_i'][i]:>8}"
                        f"{improper_types_coeffs['element_j'][i]:>8}"
                        f"{improper_types_coeffs['element_k'][i]:>8}"
                        f"{improper_types_coeffs['element_l'][i]:>8}"
                        f"     2 "
                        f"{improper_types_coeffs['theta0_deg'][i]:>8.1f}"
                        f"{improper_types_coeffs['k_eff_gromacs'][i]:>15.10f}\n")
                file.write(line)



write_bonded_info(ffbonded_name=f'{folder_name}/ffbonded.itp')

In [24]:
def write_molecule_itp(molecule_itp_name=f'{folder_name}/conf.itp',resname='UNL',angle_style=angle_style):
    if angle_style=='fourier':
        ang_num=2
    else:
        ang_num=1
        
    with open(molecule_itp_name, 'w') as file:
        # Write header
        file.write(f''';Created using lmp2gro

[ moleculetype ]
; name       nrexcl
{resname}               3
''')
        file.write(f'''[ atoms ]
;  nr   type  resnr  residu  atom   cgnr    charge
''')
        
        for i in range(atom_count):
                line = (f"{int(atom_data['atom_id'][i]):>5}"
                        f"{atom_data['gro_type'][i]:>5}"
                        f"     1 "
                        f"{resname:<5}"
                        f"{atom_data['element'][i].replace(' ',''):>5}"
                        f"{int(atom_data['atom_id'][i]):<5}"
                        f"{int(atom_data['atom_id'][i]):>5}"
                        f"{atom_data['charge'][i]:>15.6f}\n")
                file.write(line)

        file.write(f'''
[ bonds ]
;     ai      aj   func 
''')
        
        for i in range(bond_count):
                line = (f"{int(bond_data['ai'][i]):>8}"
                        f"{int(bond_data['aj'][i]):>8}"
                        f"     1 \n")
                file.write(line)


        file.write(f'''
[ angles ]
;     ai      aj      ak   func 
''')
        
        for i in range(angle_count):
                line = (f"{int(angle_data['ai'][i]):>8}"
                        f"{int(angle_data['aj'][i]):>8}"
                        f"{int(angle_data['ak'][i]):>8}"
                        f"     {ang_num} \n")
                file.write(line)

        if dihedral_count !=0:
            file.write(f'''
[ dihedrals ]
;     ai      aj      ak      al   func  
''')
            
            for i in range(dihedral_count):
                    line = (f"{int(dihedral_data['ai'][i]):>8}"
                            f"{int(dihedral_data['aj'][i]):>8}"
                            f"{int(dihedral_data['ak'][i]):>8}"
                            f"{int(dihedral_data['al'][i]):>8}"
                            f"     9 \n")
                    file.write(line)

        if improper_count !=0:        
            file.write(f'''
[ dihedrals ]
;     ai      aj      ak      al   func  
''')
            
            for i in range(improper_count):
                    line = (f"{int(improper_data['ai'][i]):>8}"
                            f"{int(improper_data['aj'][i]):>8}"
                            f"{int(improper_data['ak'][i]):>8}"
                            f"{int(improper_data['al'][i]):>8}"
                            f"     2 \n")
                    file.write(line)


write_molecule_itp(molecule_itp_name=f'{folder_name}/conf.itp')

In [ ]:
def write_atomtypes(ff_types=ff_types, atom_types=atom_types, atomtypes_name=f'{folder_name}/atomtypes.itp'):
    with open(atomtypes_name, 'w') as file:
        # Write header
        file.write(f''';Created using lmp2gro

[ atomtypes ]
;name     at.nr      mass  charge   ptype     sigma   epsilon
''')

        # Write atom types data
        for i in range(atom_types):
            line = (f"{ff_types['type'][i]:<5}"  # Left-aligned in 5 spaces
                    f"{ff_types['atomic_number'][i]:>10}"  # Right-aligned in 10 spaces
                    f"{ff_types['masses'][i]:>10.4f}"  # Right-aligned, 10 spaces, 4 decimals
                    f"  0.0000     A  "
                    f"{ff_types['gro_sigma'][i]:>10.4f}"  # Right-aligned, 10 spaces, 4 decimals
                    f"{ff_types['gro_epsilon'][i]:>10.4f}\n")  # Right-aligned, 10 spaces, 4 decimals
            file.write(line)

write_atomtypes()

In [26]:
def write_gro_file(gro_name=f'{folder_name}/conf.gro',resname='UNL'):
    with open(gro_name, 'w') as file:
        file.write(f'''Created using lmp2gro\n''')
        file.write(f"{atom_count:>5}\n")

        for i in range(atom_count):
            line = (f"{int(atom_data['mol_id'][i]):>5}"
                    f"{resname:<5}"
                    f"{atom_data['element'][i]:>5}"
                    f"{int(atom_data['atom_id'][i]):>5}"
                    f"{atom_data['x'][i]/10:>8.3f}"
                    f"{atom_data['y'][i]/10:>8.3f}"
                    f"{atom_data['z'][i]/10:>8.3f}\n")
            file.write(line)

        file.write(extract_box_params())

write_gro_file(gro_name=f'{folder_name}/conf.gro',resname='UNL')

In [27]:
def write_example_topology(topology_name=f'{folder_name}/topol.top'):
    with open(topology_name, 'w') as file:
        # Write header
        file.write(f''';Created using lmp2gro
;
;	Example topology file
;
[ defaults ]
; nbfunc        comb-rule       gen-pairs       fudgeLJ fudgeQQ
  1             3               yes              1.0     0.0

; The force field files to be included
#include "atomtypes.itp"
#include "ffbonded.itp"
#include "conf.itp"

[ system ]
Example system

[ molecules ]
UNL	1
''')


write_example_topology(topology_name=f'{folder_name}/topol.top')